# SLM Train - Custom Dataset

In [1]:

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense,  Bidirectional

# Read the text file
with open('Shakespeare_corpus.txt', 'r', encoding='utf-8') as file:
    text = file.read()

2025-09-16 11:42:15.128034: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-16 11:42:15.287250: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758003135.349846   14033 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758003135.368892   14033 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1758003135.503776   14033 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

FileNotFoundError: [Errno 2] No such file or directory: 'Shakespeare_corpus.txt'

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1
print('total words', total_words)

total words 8198


In [ ]:
input_sequences = []
for line in text.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)
print('sentence tokens', input_sequences[:20])
max_sequence_len = max([len(seq) for seq in input_sequences])
print('max sentence length', max_sequence_len)
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

sentence tokens [[4, 130], [4, 130, 34], [4, 130, 34, 37], [4, 130, 34, 37, 14], [4, 130, 34, 37, 14, 215], [4, 130, 34, 37, 14, 215, 1], [4, 130, 34, 37, 14, 215, 1, 210], [4, 130, 34, 37, 14, 215, 1, 210, 3], [4, 130, 34, 37, 14, 215, 1, 210, 3, 16], [4, 130, 34, 37, 14, 215, 1, 210, 3, 16, 1558], [4, 130, 34, 37, 14, 215, 1, 210, 3, 16, 1558, 115], [4, 130, 34, 37, 14, 215, 1, 210, 3, 16, 1558, 115, 35], [3184, 36], [3184, 36, 275], [3184, 36, 275, 102], [3184, 36, 275, 102, 94], [3184, 36, 275, 102, 94, 204], [3184, 36, 275, 102, 94, 204, 7], [3184, 36, 275, 102, 94, 204, 7, 13], [3184, 36, 275, 102, 94, 204, 7, 13, 143]]
max sentence length 18


In [ ]:
xs = input_sequences[:, :-1]
ys = input_sequences[:, -1]
print(len(xs))
print(len(ys))

96247
96247


In [ ]:
y = np.array(tf.keras.utils.to_categorical(ys, num_classes=total_words))

In [ ]:
# embedding_index = {}
# with open('glove.6B.100d.txt', encoding='utf-8') as f:
#     for line in f:
#         values = line.split()
#         word = values[0]
#         coefs = np.asarray(values[1:], dtype='float32')
#         embedding_index[word] = coefs

# embedding_dim = 100
# embedding_matrix = np.zeros((total_words, embedding_dim))
# for word, i in tokenizer.word_index.items():
#     if word in embedding_index:
#         embedding_matrix[i] = embedding_index[word]

In [ ]:
# model = Sequential()
# model.add(Embedding(total_words, 100, input_length=max_sequence_len-1))
# # model.add(Embedding(input_dim=total_words,
# #                     output_dim=embedding_dim,
# #                     weights=[embedding_matrix],
# #                     input_length=max_sequence_len-1,
# #                     trainable=False))  # freeze pretrained embeddings
# model.add(LSTM(150, return_sequences=True))
# model.add(LSTM(100))
# model.add(Dense(total_words, activation='softmax'))
# print(model.summary())


model = Sequential()
model.add(Embedding(total_words, 100, input_length = max_sequence_len - 1))
# 100 * 27k matrix --> weights
model.add(Bidirectional(LSTM(150)))
model.add(Dense(total_words, activation = 'softmax'))
model.compile(loss = 'categorical_crossentropy', optimizer = 'adam', metrics = ['accuracy'])

2025-09-16 11:37:55.925124: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [ ]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(xs, y, epochs=10, verbose=1)

: 

In [ ]:
model.save('my_text_generator_model.h5')

# Test

In [5]:

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.models import load_model

# Read the text file
with open('sherlock-holm.es_stories_plain-text_advs.txt', 'r', encoding='utf-8') as file:
    text1 = file.read()

tokenizer = Tokenizer()
tokenizer.fit_on_texts([text1])
total_words = len(tokenizer.word_index) + 1
print(total_words)

input_sequences = []
for line in text1.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)
max_sequence_len = max([len(seq) for seq in input_sequences])


8198


In [6]:
my_model = load_model('my_text_generator_model.h5')

with open('A Study In Scarlet.txt') as f:
  text = f.read()
test_text = text[5000:10000]
# print(test_text)

seed_text = """In the year 1878 I took my degree of Doctor of Medicine of the
     University of London, and proceeded to Netley to go through the
     course prescribed for surgeons in the army. Having completed my
     studies there, I was duly attached to the Fifth Northumberland
     Fusiliers as Assistant Surgeon. The regiment was stationed in India
     at the time, and before I could join it, the second Afghan war had
     broken out. On landing at Bombay, I learned that my corps had
     advanced through the passes, and was already deep in the enemy's
     country. I followed, however, with many other officers who were in
     the same situation as myself, and succeeded in reaching Candahar in
     safety, where I found my regiment, and at once entered upon my new
     duties."""

token_list_test = tokenizer.texts_to_sequences([test_text[:]])[0]
token_list_seed = tokenizer.texts_to_sequences([seed_text[:]])[0]
# print(len(token_list_seed))

count_fail = 0
pred_words = []
for k in range(len(token_list_test)):
    temp =  token_list_seed
    token_list_pad = pad_sequences([token_list_seed], maxlen=max_sequence_len-1, padding='pre')
    predicted = np.argmax(my_model.predict(token_list_pad, verbose = 0), axis=-1)
    if predicted[0] == token_list_test[k]:
        temp.append(predicted[0])
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        # print(output_word)
        pred_words.append(output_word)
    else:
        # print('--')
        count_fail+=1
        temp.append(token_list_test[k])
    token_list_seed = temp[1:]    

print('fail', count_fail)
print('pass', len(pred_words), count_fail+len(pred_words))
print(pred_words)

ValueError: Unrecognized keyword arguments passed to LSTM: {'time_major': False}

# LLM - GPT Model

In [1]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import torch
import string
from sentence_transformers import SentenceTransformer, util
import re
from collections import Counter
import numpy as np

/home/sysad/.local/lib/python3.10/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(
2025-09-18 14:48:46.351404: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-18 14:48:46.359652: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758187126.371339   27382 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E

In [2]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()
model11 = SentenceTransformer('all-MiniLM-L6-v2')

In [4]:
with open('test_file_1.txt') as f:
# with open('Datasets/sherlock_holmes/novels/The Hound of the Baskervilles.txt') as f:
  test_text = f.read()
text_words = re.findall(r"\w+|[^\w\s]", test_text, re.UNICODE)


no_seed = 50
seed_text = " ".join(text_words[:no_seed])
print('seed text: ', seed_text)

input_test_words = text_words[no_seed:]

seed text:  In the digital age , privacy has become one of the most debated and critical issues . With every click , swipe , and search , individuals leave behind a trail of data . This data , often collected without explicit consent , is used by corporations , advertisers ,


In [9]:
def Transmitter_coding(seed_text, input_text):
    tx_words = []
    for k in range(len(input_text)):
        original_tokens = seed_text.split()
        actual = input_text[k]
        if len(actual) < 2:
            new_prompt = ' '.join(original_tokens[1:] + [actual.strip()])
            tx_words.append(actual)
        else:
            input_ids = tokenizer.encode(seed_text, return_tensors='pt')
            with torch.no_grad():
                output = model.generate(
                    input_ids,
                    max_new_tokens=1,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id)
            predicted_token = output[0][input_ids.shape[-1]:]
            pred = tokenizer.decode(predicted_token, skip_special_tokens=True).strip()
            pred_clean = re.sub(r"[^\w']+", "", pred)
            actual_clean = re.sub(r"[^\w']+", "", actual)
            embeddings = model11.encode([pred_clean, actual_clean], convert_to_tensor=True)
            similarity = util.cos_sim(embeddings[0], embeddings[1])
            if similarity.item() >= 0.7:
                new_prompt = ' '.join(original_tokens[1:] + [pred.strip()])
                tx_words.append('$')
            else:
                new_prompt = ' '.join(original_tokens[1:] + [actual.strip()])
                tx_words.append(actual)
        seed_text = new_prompt
    tx_sentece = ' '.join(tx_words)
    return tx_sentece, seed_text

no_test_words = 100
frame = 0
test_text_words = input_test_words[frame*no_test_words:(frame+1)*no_test_words]
print('test words', test_text_words)

final_tx, seed_tx_next = Transmitter_coding(seed_text, test_text_words)
# print('curr seed text: ', seed_text)
print('tx final: ', final_tx)
# print('next seed text: ', seed_tx_next)

print(len(final_tx))

test words ['and', 'sometimes', 'even', 'governments', '.', 'While', 'digital', 'tools', 'offer', 'convenience', 'and', 'personalization', ',', 'they', 'also', 'raise', 'concerns', 'about', 'surveillance', ',', 'data', 'breaches', ',', 'and', 'identity', 'theft', '.', 'People', 'are', 'increasingly', 'aware', 'of', 'how', 'their', 'personal', 'information', 'is', 'being', 'tracked', ',', 'stored', ',', 'and', 'monetized', '.', 'As', 'a', 'result', ',', 'calls', 'for', 'stronger', 'privacy', 'regulations', ',', 'ethical', 'data', 'practices', ',', 'and', 'transparent', 'technology', 'use', 'are', 'gaining', 'momentum', 'around', 'the', 'world', ',', 'prompting', 'urgent', 'discussions', '.', 'Digital', 'privacy', 'has', 'become', 'a', 'defining', 'concern', 'of', 'the', '21st', 'century', ',', 'shaping', 'the', 'way', 'individuals', ',', 'businesses', ',', 'and', 'governments', 'interact', 'online', '.', 'As', 'our']
tx final:  $ sometimes $ $ . While digital tools offer convenience $ p

In [10]:
def Receiver_Model(seed_text, rx_text):
    rx_words = re.findall(r"\w+|[^\w\s]", rx_text, re.UNICODE)
    rx_collect = []
    print('Decoding...')
    for k in range(len(rx_words)):
        actual = rx_words[k]
        original_tokens = seed_text.split()
        if actual != '$':
            pred1 = actual
            new_prompt = ' '.join(original_tokens[1:] + [pred1.strip()])
            rx_collect.append(pred1)
        else:
            input_ids = tokenizer.encode(seed_text, return_tensors='pt')
            with torch.no_grad():
                output = model.generate(
                    input_ids,
                    max_new_tokens=1,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id)
            predicted_token = output[0][input_ids.shape[-1]:]
            pred = tokenizer.decode(predicted_token, skip_special_tokens=True).strip()
            new_prompt = ' '.join(original_tokens[1:] + [pred.strip()])
            rx_collect.append(' {'+ pred+'} ')
        seed_text = new_prompt
    rx_decoded = ' '.join(rx_collect)
    return rx_decoded, seed_text

final_decoded, seed_rx_next = Receiver_Model(seed_text, final_tx)
# print('curr seed text: ', seed_text)
print('rx final: ', final_decoded)
# print('next seed text: ', seed_rx_next)

Decoding...
rx final:   {and}  sometimes  {even}   {government}  . While digital tools offer convenience  {and}  personalization ,  {they}   {also}  raise concerns  {about}  surveillance , data breaches ,  {and}  identity  {theft}  . People are  {increasingly}  aware  {of}  how  {their}   {personal}   {information}   {is}   {being}  tracked , stored ,  {and}  monetized . As a  {result}  , calls  {for}  stronger  {privacy}  regulations , ethical data practices ,  {and}  transparent technology use  {are}  gaining momentum around  {the}   {world}  , prompting urgent discussions . Digital privacy has become a defining concern  {of}   {the}  21st  {century}  , shaping  {the}   {way}  individuals ,  {businesses}  ,  {and}   {governments}   {interact}  online . As our
